# DiffuLLaMA POS fixed-12 CPU completion

Uses existing extraction only. It screens all heads at progress `.50` for early/middle/late, confirms a frozen 12 candidates per depth on held-out seeds, and measures confirmed high/low heads at `.25/.50/.75`. No model is loaded. Expected CPU time: 6.5–7.5 hours; hard cap: 8 hours.

In [ ]:
import json, os, shutil, subprocess
from pathlib import Path
from google.colab import drive

REPOSITORY = 'https://github.com/Dabsoysauce/latentrelationsondlm.git'
BRANCH = 'optimize/adaptive-pos-cpu-under-3h'
PROTOCOL_COMMIT = '3da9dede1b43308c0583083c9ec70b6139134d01'
RUN_ID = 'paper-restoration-v1-diffullama-pos-token-class-linear-probes'
RESULTS_ROOT = Path('/content/drive/MyDrive/dlmrel-paper-results (1)/diffullama')
RUN_DIR = RESULTS_ROOT / 'exploratory_extensions/diffullama_7b/ewt/pos_token_class_linear_probes' / RUN_ID
CHECKOUT = Path('/content/latentrelationsondlm-pos12')
LOCAL_CACHE = Path('/content/dlmrel-pos-cache')
TOTAL_BUDGET_SECONDS = 8 * 60 * 60
VALIDATION_RESERVE_SECONDS = 15 * 60

def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=cwd, check=True, env=os.environ.copy())

os.environ.update({'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1'})
drive.mount('/content/drive')

In [ ]:
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
run('git', 'clone', '--filter=blob:none', '--no-checkout', REPOSITORY, CHECKOUT)
run('git', 'fetch', 'origin', BRANCH, cwd=CHECKOUT)
run('git', 'checkout', '--detach', PROTOCOL_COMMIT, cwd=CHECKOUT)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=CHECKOUT, text=True).strip()
assert head == PROTOCOL_COMMIT, (head, PROTOCOL_COMMIT)
run('python', '-m', 'pip', 'install', '-q', '-e', '.[dev]', cwd=CHECKOUT)

In [ ]:
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Completed DiffuLLaMA extraction run is missing: {RUN_DIR}')
feature_parquets = list((RUN_DIR / 'checkpoints').glob('paper-pos-*-features*.parquet'))
if len(feature_parquets) != 180:
    raise RuntimeError(f'Expected 180 extracted feature chunks; found {len(feature_parquets)}. Do not start fitting.')
logical_cpus = os.cpu_count() or 1
worker_counts = [n for n in (6, 8, 10, 12) if n <= logical_cpus] or [max(1, logical_cpus)]
print(json.dumps({'run_dir': str(RUN_DIR), 'feature_parquets': len(feature_parquets), 'logical_cpus': logical_cpus, 'workers_to_benchmark': worker_counts, 'free_gib': round(shutil.disk_usage('/content').free / 2**30, 2)}, indent=2))
run('python', '-m', 'pytest', '-q', 'tests/test_paper_pos_adaptive_12.py', cwd=CHECKOUT)

In [ ]:
run('python', '-m', 'dlmrel.cli', 'pos-fit-adaptive-12', '--run-dir', RUN_DIR, '--local-cache', LOCAL_CACHE, '--budget-seconds', str(TOTAL_BUDGET_SECONDS), '--validation-reserve-seconds', str(VALIDATION_RESERVE_SECONDS), '--worker-counts', *map(str, worker_counts), cwd=CHECKOUT)

In [ ]:
run('python', '-m', 'dlmrel.cli', 'validate-pos-adaptive-12', '--run-dir', RUN_DIR, cwd=CHECKOUT)
OUTPUT = RUN_DIR / 'pos_adaptive_12'
manifest = json.loads((OUTPUT / 'adaptive_manifest.json').read_text())
assert manifest['status'] == 'confirmed'
print(json.dumps({'status': manifest['status'], 'elapsed_hours': manifest['elapsed_seconds'] / 3600, 'workers': manifest['workers'], 'choices': str(OUTPUT / 'pos_head_choices.csv'), 'trajectories': str(OUTPUT / 'selected_head_progress_trajectories.csv')}, indent=2))

## DiffuLLaMA-7B matched POS-head causal ablation (GPU follow-on)

Run this section only after the preceding POS validation reports `confirmed`.
Change Colab to an **A100 High-RAM** runtime, rerun only the setup/checkout cells
at the top (leave `RUN_EXTRACTION = False` in the Dream notebook), and then run
the two cells below. The first cell fails closed unless the confirmed fixed-12
rankings and the model-specific relation-selection lock are both valid. The
second cell resumes the canonical matched ablation and skips it if it is already
complete. It never reruns POS extraction.

In [ ]:
from google.colab import userdata

# Revalidate the exact ranking bundle before allowing model inference.
run('python', '-m', 'dlmrel.cli', 'validate-pos-adaptive-12', '--run-dir', RUN_DIR, cwd=CHECKOUT)
OUTPUT = RUN_DIR / 'pos_adaptive_12'
manifest = json.loads((OUTPUT / 'adaptive_manifest.json').read_text())
if manifest.get('status') != 'confirmed' or manifest.get('matched_causal_ablation_allowed') is not True:
    raise RuntimeError('Fixed-12 POS rankings have not passed held-out confirmation.')
os.environ['DLMREL_POS_HEAD_RANKINGS'] = str(OUTPUT)

# The earlier POS cells deliberately hide the GPU. Re-enable it only here.
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'Switch Colab to an A100 High-RAM runtime, rerun setup/checkout, '
        'then rerun this cell.'
    )

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets before running the 7B-model ablation.')
os.environ['HF_TOKEN'] = hf_token
del hf_token

MODEL_ID = 'diffullama_7b'
MODEL_CONFIG = 'configs/models/diffullama_7b.yaml'
RUN_PREFIX = 'paper-restoration-v1-diffullama'
SELECTION_RUN = (
    RESULTS_ROOT / 'confirmatory_ewt' / MODEL_ID / 'ewt'
    / 'relation_head_receiver_prediction' / f'{RUN_PREFIX}-relation-selection'
)
LOCK_DIR = SELECTION_RUN / 'selection-locks'
if not (LOCK_DIR / 'selection_bundle.json').is_file():
    raise FileNotFoundError(
        f'Required model-specific relation-selection lock is missing: {LOCK_DIR}. '
        'Complete the relation-selection experiment before head ablation.'
    )

run('python', '-m', 'pip', 'install', '-q', '-r', 'requirements/diffullama.txt', cwd=CHECKOUT)
run(
    'python', '-m', 'dlmrel.cli', 'validate-selection-locks',
    '--model', MODEL_CONFIG,
    '--dataset', 'configs/datasets/ewt.yaml',
    '--experiment', 'configs/experiments/relation_head_receiver_prediction.yaml',
    '--selection-lock', LOCK_DIR,
    cwd=CHECKOUT,
)
print(json.dumps({
    'gpu': torch.cuda.get_device_name(0),
    'pos_rankings': str(OUTPUT),
    'selection_lock': str(LOCK_DIR),
}, indent=2))

In [ ]:
ABLATION_RUN_ID = f'{RUN_PREFIX}-matched-relation-head-ablation'
ABLATION_RUN = (
    RESULTS_ROOT / 'exploratory_extensions' / MODEL_ID / 'ewt'
    / 'matched_relation_head_ablation' / ABLATION_RUN_ID
)
summary_path = ABLATION_RUN / 'summary.json'
already_complete = False
if summary_path.is_file():
    already_complete = json.loads(summary_path.read_text()).get('completion_status') == 'complete'

if already_complete:
    print('Already complete; skipping model inference:', ABLATION_RUN)
else:
    run(
        'python', '-m', 'dlmrel.cli', 'run',
        '--model', MODEL_CONFIG,
        '--dataset', 'configs/datasets/ewt.yaml',
        '--experiment', 'configs/experiments/matched_relation_head_ablation.yaml',
        '--results', RESULTS_ROOT,
        '--run-id', ABLATION_RUN_ID,
        '--resume',
        '--selection-lock', LOCK_DIR,
        '--timestep-batch-size', '8',
        '--native-batch-size', '8',
        '--intervention-batch-size', '8',
        '--sentence-batch-size', '8',
        '--adaptive-batch-max-size', '32',
        cwd=CHECKOUT,
    )

run('python', '-m', 'dlmrel.cli', 'validate', '--run-dir', ABLATION_RUN, cwd=CHECKOUT)
summary = json.loads((ABLATION_RUN / 'summary.json').read_text())
if summary.get('completion_status') != 'complete':
    raise RuntimeError('Matched head ablation did not reach a validated complete state.')
print(json.dumps({
    'model': 'DiffuLLaMA-7B',
    'status': summary['completion_status'],
    'run_dir': str(ABLATION_RUN),
    'resumable': True,
    'pos_protocol': manifest['schema_version'],
}, indent=2))